# Notebook 04: Training — SUM Condition (Baseline)

**Agency Calculus Empirical Validation — Paper C**

Social planner objective: **R = Σu_i (utilitarian sum)**

This is the AI Economist default. Permits unbounded compensation:
gains for high-utility agents can offset losses for low-utility agents.

**Predicted outcomes:**
- Floor utility: declines over training (sacrificed for aggregate)
- Gini coefficient: increases (inequality grows)
- Tax on floor agent: extractive (planner taxes floor to maximize sum)
- Total utility: highest of three conditions

5 seeds × 10M steps each. Expected runtime: ~2-4 hours per seed on Colab GPU.

In [ ]:
# ── Environment check ─────────────────────────────────────────────────────
# If ai_economist is missing, run notebook 01 first (it handles installation
# and the required kernel restart).
import sys, os

try:
    import ai_economist  # noqa: F401
except ModuleNotFoundError:
    raise SystemExit(
        "\n❌  ai_economist not found. Run notebook 01_setup_and_test first,\n"
        "    restart the kernel, then return here."
    )

# Add src/ to path
for candidate in [
    '/content/ac-validation/src',
    os.path.join(os.getcwd(), '..', 'src'),
    os.path.join(os.getcwd(), 'src'),
]:
    if os.path.exists(candidate) and candidate not in sys.path:
        sys.path.insert(0, candidate)
        print(f'src on path: {candidate}')
        break


In [ ]:
import sys, os
import numpy as np

sys.path.insert(0, os.path.join(os.getcwd(), '..', 'src'))

from training import run_condition, run_training, TOTAL_TIMESTEPS, N_SEEDS
from metrics import MetricsLogger
import matplotlib.pyplot as plt

CONDITION = 'sum'
RESULTS_DIR = '../results'
os.makedirs(RESULTS_DIR, exist_ok=True)
print(f'Condition: {CONDITION}')
print(f'Total timesteps per seed: {TOTAL_TIMESTEPS:,}')
print(f'Number of seeds: {N_SEEDS}')

## Option A: Full Training (5 seeds × 10M steps)

Recommended for Kaggle (30 GPU hrs/week). Run each seed in a separate session.

In [ ]:
# Run a single seed (set SEED = 0, 1, 2, 3, 4 across separate sessions)
SEED = 0  # Change this for each session

# Check if already completed
save_path = f'{RESULTS_DIR}/{CONDITION}_seed{SEED}_metrics.npz'
if os.path.exists(save_path):
    print(f'Seed {SEED} already complete: {save_path}')
    print('Load it with: MetricsLogger.load(save_path)')
else:
    print(f'Starting training: condition={CONDITION}, seed={SEED}')
    logger = run_training(
        condition=CONDITION,
        seed=SEED,
        total_timesteps=TOTAL_TIMESTEPS,
        results_dir=RESULTS_DIR,
    )
    print(f'Training complete. Saved to {save_path}')

## Option B: Short Debug Run (1M steps, 1 seed)

Use this to verify the training loop works before committing to full runs.

In [ ]:
# DEBUG: short run to verify setup
DEBUG_STEPS = 1_000_000

# Uncomment to run:
# logger_debug = run_training(
#     condition=CONDITION,
#     seed=99,  # use seed 99 to not overwrite real results
#     total_timesteps=DEBUG_STEPS,
#     results_dir=RESULTS_DIR,
# )
print('Debug run commented out. Uncomment to verify.')

## Inline Monitoring: Plot Progress During Training

In [ ]:
def plot_seed_progress(condition: str, seed: int, results_dir: str = RESULTS_DIR):
    """Load and plot metrics for a completed seed."""
    path = f'{results_dir}/{condition}_seed{seed}_metrics.npz'
    if not os.path.exists(path):
        print(f'Not found: {path}')
        return
    
    logger = MetricsLogger.load(path)
    arrays = logger.to_arrays()
    steps = arrays.get('step', np.array([]))
    
    metrics_to_plot = ['floor_utility_mean', 'total_utility_mean', 'gini_wealth_mean']
    fig, axes = plt.subplots(1, 3, figsize=(14, 4))
    
    for ax, metric in zip(axes, metrics_to_plot):
        if metric in arrays:
            ax.plot(steps, arrays[metric], color='#e74c3c', linewidth=2)
            ax.set_xlabel('Training Steps')
            ax.set_ylabel(metric.replace('_', ' '))
            ax.set_title(metric)
            ax.spines['top'].set_visible(False)
            ax.spines['right'].set_visible(False)
    
    plt.suptitle(f'SUM Condition — Seed {seed}', y=1.02)
    plt.tight_layout()
    plt.show()

# Plot any completed seeds
for s in range(N_SEEDS):
    plot_seed_progress(CONDITION, s)

## Notes on Expected Results

Under SUM, the planner maximizes total coin. Since skills are Pareto-distributed,
high-skill agents earn much more. The optimal SUM policy extracts from low-skill
(floor) agents and redistributes upward, or simply ignores them.

Expected trajectory:
- Steps 0-2M: All agents improve (coordination phase)
- Steps 2M-10M: Gini rises, floor utility plateaus or declines
- Final: max total utility but high Gini and low floor utility

If floor utility rises monotonically, check that the planner reward is correctly
set to SUM and not accidentally using JAM or NASH.

**Next:** Notebook 05 — NASH condition.